In [1]:
!pip install torchreid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=f4f5bacf0ebaa002575312d513ef48cd179b0ba74e02602b2e67bac3940d47f5
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from sklearn.metrics import average_precision_score
import torchreid
from torchreid import utils

print("Setup selesai!")

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
2026-06-02 22:46:54.565626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780440414.788028      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780440414.850304      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780440415.342787      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780440415.342836      58 computation_placer.cc:17

Setup selesai!


In [3]:
# ============================================================
# Definisi 3 konfigurasi model
# ============================================================
MODEL_CONFIGS = [
    {
        "name"       : "No Pretrain",
        "num_classes": 1000,
        "weight_url" : None,
        "weight_file": None,
    },
    {
        "name"       : "Pretrained Market-1501",
        "num_classes": 751,
        "weight_url" : "https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA",
        "weight_file": "osnet_x1_0_market1501.pth",
    },
    {
        "name"       : "Pretrained MSMT17",
        "num_classes": 1041,
        "weight_url" : "https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M",
        "weight_file": "osnet_x1_0_msmt17.pth",
    },
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def build_model_from_config(cfg):
    model = torchreid.models.build_model(
        name='osnet_x1_0',
        num_classes=cfg["num_classes"],
        pretrained=False
    )
    if cfg["weight_url"] is not None:
        utils.download_url(cfg["weight_url"], cfg["weight_file"])
        utils.load_pretrained_weights(model, cfg["weight_file"])
        print(f"Weight loaded: {cfg['weight_file']}")
    else:
        print("No pretrain — random weights digunakan")
    model = model.to(device)
    model.eval()
    return model

print("Konfigurasi model siap!")

Device: cuda
Konfigurasi model siap!


In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
def load_images_from_folder(folder_path):
    """
    Load gambar dari folder dengan format nama: XXXX_cY_fZZZZ.jpg
    Return: list of (image_path, person_id, camera_id)
    """
    data = []
    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith('.jpg'):
            continue
        
        # Parse ID dari nama file
        # Format: XXXX_cY_fZZZZ.jpg
        parts = fname.split('_')
        try:
            person_id = int(parts[0])
            camera_id = int(parts[1][1:])  # hilangkan 'c'
        except:
            continue
        
        # Skip ID -1 (junk/distractor)
        if person_id == -1:
            continue
            
        img_path = os.path.join(folder_path, fname)
        data.append((img_path, person_id, camera_id))
    
    return data

print("Fungsi siap!")

Fungsi siap!


In [6]:
def extract_features_from_paths(data_list, model, transform, device, batch_size=32):
    """
    Ekstrak fitur dari list of (img_path, person_id, camera_id)
    Return: features, labels, camera_ids
    """
    features_list = []
    labels_list   = []
    cameras_list  = []
    
    model.eval()
    with torch.no_grad():
        for i in range(0, len(data_list), batch_size):
            batch = data_list[i:i+batch_size]
            
            imgs = []
            for img_path, pid, cid in batch:
                img = Image.open(img_path).convert('RGB')
                imgs.append(transform(img))
            
            imgs = torch.stack(imgs).to(device)
            feats = model(imgs)
            features_list.append(feats.cpu().numpy())
            labels_list.extend([x[1] for x in batch])
            cameras_list.extend([x[2] for x in batch])
            
            if i % 320 == 0:
                print(f"  Processed {i}/{len(data_list)}")
    
    features = np.vstack(features_list)
    return features, labels_list, cameras_list

print("Fungsi siap!")

Fungsi siap!


In [7]:
def compute_cosine_distance(qf, gf):
    q = qf / np.linalg.norm(qf, axis=1, keepdims=True)
    g = gf / np.linalg.norm(gf, axis=1, keepdims=True)
    return 1 - np.dot(q, g.T)

def evaluate_reid(query_feats, query_labels, gallery_feats, gallery_labels):
    dist_matrix = compute_cosine_distance(query_feats, gallery_feats)
    
    num_query  = len(query_labels)
    cmc_scores = np.zeros(10)
    ap_list    = []

    for i in range(num_query):
        q_label    = query_labels[i]
        sorted_idx = np.argsort(dist_matrix[i])
        sorted_labels = [gallery_labels[j] for j in sorted_idx]
        matches    = [1 if lbl == q_label else 0 for lbl in sorted_labels]
        
        if sum(matches) == 0:
            continue
        
        for rank in range(10):
            if sum(matches[:rank+1]) > 0:
                cmc_scores[rank] += 1
        
        ap_list.append(average_precision_score(
            matches, [-d for d in dist_matrix[i][sorted_idx]]
        ))

    cmc_scores /= num_query
    mAP = np.mean(ap_list) if ap_list else 0
    return cmc_scores, mAP

print("Fungsi baseline siap!")

Fungsi baseline siap!


In [8]:
def evaluate_reid_reranking(query_feats, query_labels, gallery_feats, gallery_labels, k1=20, k2=6, lambda_value=0.3):
    # Hitung 3 distance matrix
    q_q_dist = compute_cosine_distance(query_feats, query_feats)
    q_g_dist = compute_cosine_distance(query_feats, gallery_feats)
    g_g_dist = compute_cosine_distance(gallery_feats, gallery_feats)
    
    # Re-ranking
    dist_matrix = utils.re_ranking(
        q_g_dist, q_q_dist, g_g_dist,
        k1=k1, k2=k2, lambda_value=lambda_value
    )
    
    num_query  = len(query_labels)
    cmc_scores = np.zeros(10)
    ap_list    = []

    for i in range(num_query):
        q_label    = query_labels[i]
        sorted_idx = np.argsort(dist_matrix[i])
        sorted_labels = [gallery_labels[j] for j in sorted_idx]
        matches    = [1 if lbl == q_label else 0 for lbl in sorted_labels]
        
        if sum(matches) == 0:
            continue
        
        for rank in range(10):
            if sum(matches[:rank+1]) > 0:
                cmc_scores[rank] += 1
        
        ap_list.append(average_precision_score(
            matches, [-d for d in dist_matrix[i][sorted_idx]]
        ))

    cmc_scores /= num_query
    mAP = np.mean(ap_list) if ap_list else 0
    return cmc_scores, mAP

print("Fungsi re-ranking siap!")

Fungsi re-ranking siap!


In [9]:
dataset_root = "/kaggle/input/datasets/singh96divya/wb-wob-reid-dataset/WB_WoB-ReID"
subsets      = ["with_bag", "without_bag", "both_small", "both_large"]

# Simpan hasil semua model
all_model_results = {}

for cfg in MODEL_CONFIGS:
    model_name = cfg["name"]
    print(f"\n{'#'*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'#'*70}")

    # Build model sesuai config
    model = build_model_from_config(cfg)

    all_model_results[model_name] = {}

    for subset in subsets:
        print(f"\n{'='*50}")
        print(f"  Subset: {subset}")
        print(f"{'='*50}")

        # Path
        test_path  = os.path.join(dataset_root, subset, "bounding_box_test")
        query_path = os.path.join(dataset_root, subset, "query")

        # Load data
        gallery_data = load_images_from_folder(test_path)
        query_data   = load_images_from_folder(query_path)
        print(f"Gallery: {len(gallery_data)} | Query: {len(query_data)}")

        # Ekstrak fitur
        print("Ekstrak fitur gallery...")
        g_feats, g_labels, g_cams = extract_features_from_paths(
            gallery_data, model, transform, device
        )
        print("Ekstrak fitur query...")
        q_feats, q_labels, q_cams = extract_features_from_paths(
            query_data, model, transform, device
        )

        # Evaluasi
        cmc_base, map_base = evaluate_reid(q_feats, q_labels, g_feats, g_labels)
        cmc_rr,   map_rr   = evaluate_reid_reranking(q_feats, q_labels, g_feats, g_labels)

        all_model_results[model_name][subset] = {
            "baseline" : {
                "r1" : cmc_base[0]*100, "r5" : cmc_base[4]*100,
                "r10": cmc_base[9]*100, "mAP": map_base*100
            },
            "reranking": {
                "r1" : cmc_rr[0]*100,   "r5" : cmc_rr[4]*100,
                "r10": cmc_rr[9]*100,   "mAP": map_rr*100
            },
        }

        print(f"Baseline   → R1: {cmc_base[0]*100:.2f}%  mAP: {map_base*100:.2f}%")
        print(f"Re-ranking → R1: {cmc_rr[0]*100:.2f}%  mAP: {map_rr*100:.2f}%")

    # Bebaskan VRAM setelah setiap model selesai
    del model
    torch.cuda.empty_cache()

print("\n\nSemua evaluasi selesai!")


######################################################################
  MODEL: No Pretrain
######################################################################
No pretrain — random weights digunakan

  Subset: with_bag
Gallery: 1131 | Query: 322
Ekstrak fitur gallery...
  Processed 0/1131
  Processed 320/1131
  Processed 640/1131
  Processed 960/1131
Ekstrak fitur query...
  Processed 0/322
  Processed 320/322
Baseline   → R1: 32.30%  mAP: 7.76%
Re-ranking → R1: 25.47%  mAP: 7.27%

  Subset: without_bag
Gallery: 986 | Query: 179
Ekstrak fitur gallery...
  Processed 0/986
  Processed 320/986
  Processed 640/986
  Processed 960/986
Ekstrak fitur query...
  Processed 0/179
Baseline   → R1: 49.16%  mAP: 25.15%
Re-ranking → R1: 31.84%  mAP: 20.88%

  Subset: both_small
Gallery: 2146 | Query: 475
Ekstrak fitur gallery...
  Processed 0/2146
  Processed 320/2146
  Processed 640/2146
  Processed 960/2146
  Processed 1280/2146
  Processed 1600/2146
  Processed 1920/2146
Ekstrak fitur query..

In [10]:
for model_name, model_results in all_model_results.items():
    print("\n" + "=" * 75)
    print(f"  MODEL: {model_name}".center(75))
    print("=" * 75)
    print(f"{'Subset':<15} {'Metode':<12} {'Rank-1':>8} {'Rank-5':>8} {'Rank-10':>8} {'mAP':>8}")
    print("-" * 75)

    for subset in subsets:
        r = model_results[subset]

        print(f"{subset:<15} {'Baseline':<12} "
              f"{r['baseline']['r1']:>7.2f}% "
              f"{r['baseline']['r5']:>7.2f}% "
              f"{r['baseline']['r10']:>7.2f}% "
              f"{r['baseline']['mAP']:>7.2f}%")

        print(f"{'':15} {'Re-ranking':<12} "
              f"{r['reranking']['r1']:>7.2f}% "
              f"{r['reranking']['r5']:>7.2f}% "
              f"{r['reranking']['r10']:>7.2f}% "
              f"{r['reranking']['mAP']:>7.2f}%")

        d_r1  = r['reranking']['r1']  - r['baseline']['r1']
        d_r5  = r['reranking']['r5']  - r['baseline']['r5']
        d_r10 = r['reranking']['r10'] - r['baseline']['r10']
        d_map = r['reranking']['mAP'] - r['baseline']['mAP']
        print(f"{'':15} {'Selisih':<12} "
              f"{d_r1:>+7.2f}% {d_r5:>+7.2f}% {d_r10:>+7.2f}% {d_map:>+7.2f}%")
        print("-" * 75)

    print("=" * 75)


                              MODEL: No Pretrain                           
Subset          Metode         Rank-1   Rank-5  Rank-10      mAP
---------------------------------------------------------------------------
with_bag        Baseline       32.30%   46.58%   57.45%    7.76%
                Re-ranking     25.47%   40.99%   50.00%    7.27%
                Selisih        -6.83%   -5.59%   -7.45%   -0.48%
---------------------------------------------------------------------------
without_bag     Baseline       49.16%   69.27%   74.86%   25.15%
                Re-ranking     31.84%   59.78%   69.27%   20.88%
                Selisih       -17.32%   -9.50%   -5.59%   -4.26%
---------------------------------------------------------------------------
both_small      Baseline       34.11%   49.68%   57.89%   12.06%
                Re-ranking     24.63%   44.63%   51.58%   10.26%
                Selisih        -9.47%   -5.05%   -6.32%   -1.80%
---------------------------------------------

In [11]:
import gradio as gr
from PIL import Image
import numpy as np
import pandas as pd

# ============================================================
# 1. Precompute gallery cache (sama seperti sebelumnya)
# ============================================================

SUBSET_LIST     = ["with_bag", "without_bag", "both_small", "both_large"]
MODEL_NAME_LIST = ["No Pretrain", "Pretrained Market-1501", "Pretrained MSMT17"]

print("Membangun gallery cache...")
models_dict = {}
for cfg in MODEL_CONFIGS:
    models_dict[cfg["name"]] = build_model_from_config(cfg)

gallery_cache = {}
for model_name, model in models_dict.items():
    gallery_cache[model_name] = {}
    for subset in SUBSET_LIST:
        test_path    = os.path.join(dataset_root, subset, "bounding_box_test")
        gallery_data = load_images_from_folder(test_path)
        print(f"  [{model_name}] x [{subset}] — {len(gallery_data)} gambar")
        g_feats, g_labels, g_cams = extract_features_from_paths(
            gallery_data, model, transform, device
        )
        gallery_cache[model_name][subset] = {
            "feats" : g_feats,
            "labels": g_labels,
            "paths" : [d[0] for d in gallery_data],
        }

print("\nGallery cache selesai!")


# ============================================================
# 2. Buat teks tabel evaluasi dari all_model_results
# ============================================================

def build_eval_table_md():
    """Ubah all_model_results jadi string markdown tabel."""
    lines = []
    for model_name in MODEL_NAME_LIST:
        lines.append(f"### {model_name}")
        lines.append("| Subset | Metode | Rank-1 | Rank-5 | Rank-10 | mAP |")
        lines.append("|--------|--------|--------|--------|---------|-----|")
        for subset in SUBSET_LIST:
            r  = all_model_results[model_name][subset]
            b  = r["baseline"]
            rr = r["reranking"]
            d_r1  = rr["r1"]  - b["r1"]
            d_r5  = rr["r5"]  - b["r5"]
            d_r10 = rr["r10"] - b["r10"]
            d_map = rr["mAP"] - b["mAP"]
            lines.append(
                f"| {subset} | Baseline   "
                f"| {b['r1']:.2f}% | {b['r5']:.2f}% | {b['r10']:.2f}% | {b['mAP']:.2f}% |"
            )
            lines.append(
                f"|  | Re-ranking "
                f"| {rr['r1']:.2f}% | {rr['r5']:.2f}% | {rr['r10']:.2f}% | {rr['mAP']:.2f}% |"
            )
            lines.append(
                f"|  | **Selisih** "
                f"| {d_r1:+.2f}% | {d_r5:+.2f}% | {d_r10:+.2f}% | {d_map:+.2f}% |"
            )
        lines.append("")
    return "\n".join(lines)

EVAL_TABLE_MD = build_eval_table_md()


# ============================================================
# 3. Fungsi utama
# ============================================================

TOP_K = 10

def get_query_id_from_upload(query_image_path):
    """Ambil person ID dari nama file yang diupload."""
    if query_image_path is None:
        return None
    fname = os.path.basename(query_image_path)
    try:
        person_id = int(fname.split("_")[0])
    except:
        person_id = None
    return person_id

def reid_demo(query_image_path, model_name, subset):
    if query_image_path is None:
        return [], [], "", ""

    # --- Ambil query ID dari nama file ---
    query_pid  = get_query_id_from_upload(query_image_path)
    query_info = f"**ID Gambar Query: {query_pid}**" if query_pid is not None else "ID tidak dikenali"

    # --- Buka gambar & ekstrak fitur ---
    query_image = Image.open(query_image_path).convert("RGB")
    model       = models_dict[model_name]
    img_t       = transform(query_image).unsqueeze(0).to(device)
    with torch.no_grad():
        q_feat = model(img_t).cpu().numpy()

    # --- Ambil gallery dari cache ---
    cache    = gallery_cache[model_name][subset]
    g_feats  = cache["feats"]
    g_labels = cache["labels"]
    g_paths  = cache["paths"]

    # --- Baseline: cosine distance ---
    q_norm   = q_feat  / np.linalg.norm(q_feat,  axis=1, keepdims=True)
    g_norm   = g_feats / np.linalg.norm(g_feats, axis=1, keepdims=True)
    cos_dist = 1 - np.dot(q_norm, g_norm.T).squeeze(0)
    base_idx = np.argsort(cos_dist)[:TOP_K]

    # --- Re-ranking: k-reciprocal ---
    q_q_dist = compute_cosine_distance(q_feat,  q_feat)
    q_g_dist = compute_cosine_distance(q_feat,  g_feats)
    g_g_dist = compute_cosine_distance(g_feats, g_feats)
    rr_dist  = utils.re_ranking(
        q_g_dist, q_q_dist, g_g_dist,
        k1=20, k2=6, lambda_value=0.3
    ).squeeze(0)
    rr_idx   = np.argsort(rr_dist)[:TOP_K]

    # --- Susun output gambar & tabel ---
    def make_outputs(indices, dist_arr):
        images = []
        rows   = []
        for rank, idx in enumerate(indices):
            img      = Image.open(g_paths[idx]).convert("RGB")
            pid      = g_labels[idx]
            score    = dist_arr[idx]
            is_match = "✅ True" if pid == query_pid else "❌ False"
            images.append((img, f"Rank {rank+1} | ID:{pid} | dist:{score:.3f}"))
            rows.append({
                "Rank"    : rank + 1,
                "ID"      : pid,
                "Distance": round(float(score), 4),
                "Match?"  : is_match,
            })
        df = pd.DataFrame(rows)
        return images, df

    base_imgs, base_df = make_outputs(base_idx, cos_dist)
    rr_imgs,   rr_df   = make_outputs(rr_idx,   rr_dist)

    # Format dataframe jadi markdown
    base_md = base_df.to_markdown(index=False)
    rr_md   = rr_df.to_markdown(index=False)

    return base_imgs, rr_imgs, base_md, rr_md, query_info


# ============================================================
# 4. Bangun antarmuka Gradio
# ============================================================

with gr.Blocks(title="Person Re-ID Demo") as demo:

    # --- Header & Tabel Evaluasi ---
    gr.Markdown("# 🔍 Person Re-Identification Demo")
    gr.Markdown("**Dataset:** WB/WoB-ReID &nbsp;|&nbsp; **Model:** OSNet x1.0")

    with gr.Accordion("📊 Lihat Hasil Evaluasi Lengkap (Semua Model & Subset)", open=False):
        gr.Markdown(EVAL_TABLE_MD)

    gr.Markdown("---")

    # --- Input ---
    gr.Markdown("## Upload Gambar Query")
    gr.Markdown("Upload gambar dari folder `query/` dataset. Nama file harus berformat `XXXX_cY_fZZZZ.jpg`.")

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Image(type="filepath", label="Upload Gambar Query")
            query_id_md = gr.Markdown("")
            model_dd    = gr.Dropdown(
                choices=MODEL_NAME_LIST,
                value="Pretrained Market-1501",
                label="Pilih Model"
            )
            subset_dd   = gr.Dropdown(
                choices=SUBSET_LIST,
                value="with_bag",
                label="Pilih Subset Gallery"
            )
            run_btn     = gr.Button("🔎 Cari", variant="primary")

    gr.Markdown("---")

    # --- Output ---
    gr.Markdown("## Hasil Retrieval")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📋 Baseline (Cosine Distance)")
            base_gallery = gr.Gallery(
                label="Top-10 Baseline",
                columns=5,
                rows=2,
                height=400,
                show_label=False
            )
            gr.Markdown("#### Tabel Baseline")
            base_table = gr.Markdown("")

        with gr.Column():
            gr.Markdown("### ✨ Re-ranking (k-Reciprocal)")
            rr_gallery = gr.Gallery(
                label="Top-10 Re-ranking",
                columns=5,
                rows=2,
                height=400,
                show_label=False
            )
            gr.Markdown("#### Tabel Re-ranking")
            rr_table = gr.Markdown("")

    # --- Tampilkan query ID saat gambar diupload ---
    def show_query_id(filepath):
        if filepath is None:
            return ""
        fname = os.path.basename(filepath)
        try:
            pid = int(fname.split("_")[0])
            return f"**ID Gambar Query: {pid}**"
        except:
            return "⚠️ Format nama file tidak dikenali"

    query_input.change(fn=show_query_id, inputs=query_input, outputs=query_id_md)

    run_btn.click(
        fn=reid_demo,
        inputs=[query_input, model_dd, subset_dd],
        outputs=[base_gallery, rr_gallery, base_table, rr_table, query_id_md]
    )

    gr.Markdown("""
    ---
    💡 **Tips:** ✅ True = ID hasil retrieval sama dengan ID query (true match). ❌ False = beda orang.
    """)

demo.launch(share=True)

Membangun gallery cache...
No pretrain — random weights digunakan
* url="https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA"
* destination="osnet_x1_0_market1501.pth"
...100%, 9 MB, 9406 KB/s, 1 seconds passed
Successfully loaded pretrained weights from "osnet_x1_0_market1501.pth"
Weight loaded: osnet_x1_0_market1501.pth
* url="https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M"
* destination="osnet_x1_0_msmt17.pth"
...100%, 10 MB, 9339 KB/s, 1 seconds passed
Successfully loaded pretrained weights from "osnet_x1_0_msmt17.pth"
Weight loaded: osnet_x1_0_msmt17.pth
  [No Pretrain] x [with_bag] — 1131 gambar
  Processed 0/1131
  Processed 320/1131
  Processed 640/1131
  Processed 960/1131
  [No Pretrain] x [without_bag] — 986 gambar
  Processed 0/986
  Processed 320/986
  Processed 640/986
  Processed 960/986
  [No Pretrain] x [both_small] — 2146 gambar
  Processed 0/2146
  Processed 320/2146
  Processed 640/2146
  Processed 960/2146
  Processed 1280/2146
  Proc